In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Load the synthetic restaurant master dataset

PROJECT_ROOT = Path.cwd().parent

RESTAURANT_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "orders"
    / "restaurants.csv"
)

restaurants = pd.read_csv(RESTAURANT_FILE)

print(f"Dataset shape: {restaurants.shape}")
print(f"Number of restaurants: {len(restaurants):,}")

restaurants.head()

Dataset shape: (220, 8)
Number of restaurants: 220


,restaurant_id,restaurant_name,zone_id,zone_name,zone_type,restaurant_category,base_popularity,average_order_value
0,Z01_R001,Express Bites 001,Z01,Kondapur,IT_Residential,Bakery,0.666,122.27
1,Z01_R002,Express House 002,Z01,Kondapur,IT_Residential,Chinese,0.571,240.27
2,Z01_R003,Grand House 003,Z01,Kondapur,IT_Residential,Bakery,0.654,248.13
3,Z01_R004,Hyderabad House 004,Z01,Kondapur,IT_Residential,Healthy Food,0.385,326.58
4,Z01_R005,Daily Kitchen 005,Z01,Kondapur,IT_Residential,Chinese,0.922,242.82


In [4]:
# Validate missing values and duplicate restaurant IDs

print("Missing values by column:")
print(restaurants.isnull().sum())

print("\nDuplicate restaurant IDs:")
print(restaurants["restaurant_id"].duplicated().sum())

print("\nUnique restaurant IDs:")
print(restaurants["restaurant_id"].nunique())

Missing values by column:
restaurant_id          0
restaurant_name        0
zone_id                0
zone_name              0
zone_type              0
restaurant_category    0
base_popularity        0
average_order_value    0
dtype: int64

Duplicate restaurant IDs:
0

Unique restaurant IDs:
220


In [5]:
# Analyze restaurant distribution across Hyderabad zones

zone_distribution = (
    restaurants
    .groupby(["zone_id", "zone_name"])
    .size()
    .reset_index(name="restaurant_count")
    .sort_values("zone_id")
)

zone_distribution

,zone_id,zone_name,restaurant_count
0,Z01,Kondapur,40
1,Z02,Hitech City,50
2,Z03,Gachibowli,45
3,Z04,Madhapur,40
4,Z05,Kukatpally,45


In [6]:
# Analyze restaurant category distribution

category_distribution = (
    restaurants["restaurant_category"]
    .value_counts()
    .sort_index()
)

category_distribution

restaurant_category
Bakery          21
Biryani         20
Burgers         20
Cafe            28
Chinese         23
Desserts        15
Fast Food       21
Healthy Food    15
North Indian    17
Pizza           20
South Indian    20
Name: count, dtype: int64

In [7]:
# Summary statistics for restaurant characteristics

restaurants[
    ["base_popularity", "average_order_value"]
].describe()

,base_popularity,average_order_value
count,220.000000,220.000000
mean,0.562650,302.030182
std,0.210805,102.900293
min,0.061000,105.420000
25%,0.399500,220.897500
50%,0.599000,290.005000
75%,0.731250,374.960000
max,0.965000,545.150000


Validate customers

In [8]:
# Load the synthetic customer master dataset

CUSTOMER_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "orders"
    / "customers.csv"
)

customers = pd.read_csv(CUSTOMER_FILE)

print(f"Dataset shape: {customers.shape}")
print(f"Number of customers: {len(customers):,}")

customers.head()

Dataset shape: (50000, 6)
Number of customers: 50,000


,customer_id,home_zone_id,home_zone_name,customer_segment,customer_activity_score,preferred_order_value
0,C000001,Z02,Hitech City,Regular,0.553,322.28
1,C000002,Z05,Kukatpally,Regular,0.473,313.17
2,C000003,Z04,Madhapur,Occasional,0.160,281.59
3,C000004,Z03,Gachibowli,Regular,0.642,392.46
4,C000005,Z01,Kondapur,Occasional,0.152,251.88


In [9]:
# Validate missing values and duplicate customer IDs

print("Missing values by column:")
print(customers.isnull().sum())

print("\nDuplicate customer IDs:")
print(customers["customer_id"].duplicated().sum())

print("\nUnique customer IDs:")
print(customers["customer_id"].nunique())

Missing values by column:
customer_id                0
home_zone_id               0
home_zone_name             0
customer_segment           0
customer_activity_score    0
preferred_order_value      0
dtype: int64

Duplicate customer IDs:
0

Unique customer IDs:
50000


In [11]:
# Analyze customer distribution across zones

customer_zone_distribution = (
    customers
    .groupby(["home_zone_id", "home_zone_name"])
    .size()
    .reset_index(name="customer_count")
    .sort_values("home_zone_id")
)

customer_zone_distribution

,home_zone_id,home_zone_name,customer_count
0,Z01,Kondapur,9010
1,Z02,Hitech City,12121
2,Z03,Gachibowli,10936
3,Z04,Madhapur,8007
4,Z05,Kukatpally,9926


In [12]:
# Analyze customer segment distribution

customer_segment_distribution = (
    customers["customer_segment"]
    .value_counts()
    .rename_axis("customer_segment")
    .reset_index(name="customer_count")
)

customer_segment_distribution

,customer_segment,customer_count
0,Occasional,22396
1,Regular,20174
2,Frequent,7430


In [13]:
# Validate customer behavioral variables

customers[
    ["customer_activity_score", "preferred_order_value"]
].describe()

,customer_activity_score,preferred_order_value
count,50000.000000,50000.000000
mean,0.428319,296.667356
std,0.249080,87.444539
min,0.050000,150.000000
25%,0.218000,227.760000
50%,0.395000,287.890000
75%,0.613000,348.070000
max,1.000000,549.930000


In [14]:
# Compare behavioral characteristics across customer segments

segment_behavior = (
    customers
    .groupby("customer_segment")[
        ["customer_activity_score", "preferred_order_value"]
    ]
    .agg(["mean", "min", "max"])
    .round(3)
)

segment_behavior

customer_activity_score             preferred_order_value  \
                                    mean   min   max                  mean   
customer_segment                                                             
Frequent                           0.849  0.70  1.00               387.178   
Occasional                         0.200  0.05  0.35               249.519   
Regular                            0.526  0.35  0.70               315.674   

                                  
                     min     max  
customer_segment                  
Frequent          220.01  549.93  
Occasional        150.00  350.00  
Regular           180.01  450.00